In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [ ]:

# Expert

class Expert(nn.Module):
    """
    A small neural network that transforms the current state.
    """
    def __init__(self, state_dim, hidden_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, state_dim)
        )

    def forward(self, x):
        return x + self.network(x)


In [ ]:

# Dynamic Decision Graph

class DynamicDecisionGraph(nn.Module):
    """
    The graph itself, for now will use softmax routing and gets collection of experts via routing with 
    models, additionally for now all experts will be the same
    """
    def __init__(
        self,
        input_dim,
        state_dim,
        hidden_dim,
        num_classes,
        num_experts,
        connections,
        max_steps=5
    ):
        super().__init__()
        self.num_experts = num_experts
        self.max_steps = max_steps

        # Convert input into state representation
        self.input_projection = nn.Linear(
            input_dim,
            state_dim
        )

        # Make list of small experts
        self.experts = nn.ModuleList([
            Expert(
                state_dim=state_dim,
                hidden_dim=hidden_dim
            )
            for _ in range(num_experts)
        ])

        # Router
        self.router = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_experts)
        )

        # Final classifier
        self.classifier = nn.Linear(
            state_dim,
            num_classes
        )

        self.connections = connections

    # Routing

    def route(self, state, current_expert):
        """
        Produce a probability distribution over the
        available neighboring experts.
        """

        candidates = self.connections[current_expert]

        # Get scores for all experts
        all_scores = self.router(state)

        # Only retain scores for connected experts
        candidate_scores = all_scores[:, candidates]

        # Convert scores into probabilities
        probabilities = F.softmax(
            candidate_scores,
            dim=-1
        )

        return candidates, probabilities

    # Forward

    def forward(self, x, return_trajectory=False):

        # Convert input into state representation
        state = self.input_projection(x)

        # Start at expert 0, store trajectory
        current_expert = 0
        trajectory = []

        # Repeated dynamic routing
        for step in range(self.max_steps):

            # Run current
            state = self.experts[current_expert](state)

            # Compute routing probabilities
            if current_expert in self.connections:
                candidates, probabilities = self.route(
                    state,
                    current_expert
                )
            else:
                candidates = []
                probabilities = None

            # Record trajectory info
            trajectory.append({
                "expert": current_expert,
                "candidates": candidates,
                "probabilities": probabilities
            })

            # Terminate if no more
            if len(candidates) == 0:
                break

            # Select next expert
            next_index = torch.multinomial(
                probabilities[0],
                num_samples=1
            ).item()

            current_expert = candidates[next_index]

        # Predict
        logits = self.classifier(state)
        if return_trajectory:
            return logits, trajectory
        return logits

In [ ]:

# Device stuff
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Using:", device)

In [ ]:
# Training
# Note that currently using MNIST dataset that's not complicated

# Data handling (convert then normalize)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.1307,),
        (0.3081,)
    )
])

# Datasets
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)


In [ ]:

# Manual graph construction
connections = {
    0: [1, 2, 3],
    1: [2, 4],
    2: [3, 5],
    3: [1, 5],
    4: [6],
    5: [4, 6],
    6: [7],
    7: []
}

# Model
model = DynamicDecisionGraph(
    input_dim=28 * 28,
    state_dim=128,
    hidden_dim=128,
    num_classes=10,
    num_experts=8,
    connections=connections,
    max_steps=6
).to(device)

# Adam optimizer selected for its in-built regularizationa and performance
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3 # Current tentative
)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Training
epochs = 100 # odify LR as epochs increase

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        images = images.view(
            images.size(0),
            -1
        )

        # Forward pass
        logits = model(images)
        # Loss
        loss = criterion(
            logits,
            labels
        )

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Stats
        total_loss += loss.item()
        predictions = logits.argmax(dim=1)
        correct += (
            predictions == labels
        ).sum().item()
        total += labels.size(0)

    # Overall stats
    accuracy = correct / total
    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"| Loss: {total_loss / len(train_loader):.4f} "
        f"| Accuracy: {accuracy:.4f}"
    )